In [1]:
# Instalar dependências
!pip install langchain faiss-cpu sentence-transformers openai pdfminer.six unstructured[local] transformers accelerate tiktoken

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import os

BASE_DIR = Path.cwd()
DOCS_DIR = BASE_DIR / "docs"
DOCS_DIR.mkdir(exist_ok=True)

DATA_DIR = BASE_DIR / "rag_data"
DATA_DIR.mkdir(exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("DOCS_DIR:", DOCS_DIR)
print("DATA_DIR:", DATA_DIR)


BASE_DIR: c:\Temp
DOCS_DIR: c:\Temp\docs
DATA_DIR: c:\Temp\rag_data


## 📄 Carregar PDFs

In [3]:
from langchain.document_loaders import PyPDFLoader

def load_pdfs(folder: Path):
    docs = []
    for pdf in sorted(folder.glob("*.pdf")):
        print("Carregando:", pdf.name)
        loader = PyPDFLoader(str(pdf))
        pages = loader.load()
        docs.extend(pages)
    print("Total de páginas carregadas:", len(docs))
    return docs

documents = load_pdfs(DOCS_DIR)

if documents:
    print("\nPrévia:\n", documents[0].page_content[:300])


Total de páginas carregadas: 0


## ✂️ Criar chunks

In [4]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(documents)

print("Chunks criados:", len(chunks))


Chunks criados: 0


## 🔎 Embeddings e FAISS

In [9]:
def get_embeddings():
    if os.getenv("OPENAI_API_KEY"):
        from langchain.embeddings import OpenAIEmbeddings
        print("Usando OpenAIEmbeddings")
        return OpenAIEmbeddings()
    else:
        from langchain.embeddings import HuggingFaceEmbeddings
        print("Usando HuggingFaceEmbeddings (MiniLM)")
        return HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

emb = get_embeddings()

from langchain.vectorstores import FAISS

faiss_path = DATA_DIR / "faiss_index"

if chunks:
    print("Criando FAISS...")
    vectordb = FAISS.from_documents(chunks, emb)
    vectordb.save_local(str(faiss_path))
    print("FAISS salvo!")
else:
    if faiss_path.exists():
        print("Carregando FAISS existente...")
        vectordb = FAISS.load_local(str(faiss_path), emb)
    else:
        vectordb = None
        print("Nenhum PDF carregado!")

if vectordb:
    retriever = vectordb.as_retriever(search_kwargs={"k": 4})
    print("Retriever criado.")
else:
    retriever = None
    print("ERRO: vectordb não existe.")


Usando HuggingFaceEmbeddings (MiniLM)
Nenhum PDF carregado!
ERRO: vectordb não existe.


## 🧠 Criar LLM (OpenAI ou Local)

In [6]:
def get_llm():
    if os.getenv("OPENAI_API_KEY"):
        from langchain.chat_models import ChatOpenAI
        print("Usando ChatGPT (OpenAI)")
        return ChatOpenAI(temperature=0, model_name="gpt-3.5-turbo")
    else:
        from langchain.llms import HuggingFacePipeline
        from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

        model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
        print("Carregando LLM local:", model_name)

        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(model_name)

        pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=256)

        return HuggingFacePipeline(pipeline=pipe)

llm = get_llm()


Carregando LLM local: TinyLlama/TinyLlama-1.1B-Chat-v1.0


Device set to use cpu


## 🤖 Criar o Agente RAG

In [8]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

prompt_pt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
Você é um assistente especialista e deve responder SEMPRE em português do Brasil.

Use apenas o contexto abaixo. 
Se a resposta não estiver no contexto, diga claramente que não encontrou.

Contexto:
{context}

Pergunta:
{question}

Resposta em português:
"""
)

rag = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt_pt}
)

print("RAG criado com sucesso!")


ValidationError: 1 validation error for RetrievalQA
retriever
  none is not an allowed value (type=type_error.none.not_allowed)

## ❓ Fazer perguntas ao agente

In [ ]:
def ask(q):
    result = rag(q + " (responda em português)")
    
    print("\n🟢 Resposta:")
    print(result["result"])

    print("\n📄 Fontes:")
    for d in result["source_documents"]:
        print("-", d.metadata.get("source", "Documento"), "página:", d.metadata.get("page", "?"))


## 📌 CÉLULA 11 — Modo Chat (usuário digita perguntas)

In [ ]:
def chat():
    print("Digite sua pergunta (ENTER vazio para sair):")
    while True:
        q = input("\nPergunta: ").strip()
        if q == "":
            print("Encerrando chat.")
            break
        ask(q)

chat()


Digite sua pergunta (ENTER vazio para sair):
Encerrando chat.
